# Triage_HF Statistical & Graphical Analytical Model
* Started date: 02/07/2024 - 05:09 AM
* Data Management Team, Triage_HF

# 1. Dataset cleaning
We will import our dataset and perform a first cleanup to begin to understand which columns and values we are dealing with.

In [191]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime as dt
import plotly.io as pio


import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
import string

import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


#pio.renderers.default = "browser"
pd.options.mode.chained_assignment = None
pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

There are two ways to import our data set:
1. Locally: if we have the repository locally, we will use the following line

In [192]:
# Locally method
train_df = pd.read_csv('../dataset/raw/TRIAGE_2024.csv')

2. Remotely: if we want to connect the dataset from github, we must import it using the following line



**WARNING**: every time we want to use this form, we must generate the token again.

In [193]:
# Github method
# train_df = pd.read_csv('https://raw.githubusercontent.com/Adriellevy/Triage_HF/main/Data%20Analysis%20and%20Reports/dataset/raw/TRIAGE%202024.csv?token=GHSAT0AAAAAACLM5VZGUTNJJPTGADJMMUKEZOQ3JLA')

In [194]:
train_df.head()

,3+-99999|a,[ñ_MJ,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,A,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,FECHA: 01/01/2024 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01-01,LUSI,FIEBRE Y TOS,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,IV,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01-01,BANDERA,TOS,12,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.1 Columns rename

At first, you can see how certain columns are wrongly named, so I will assign a corresponding name to them:

In [195]:
cols_rename = { '[ñ_MJ': 'FECHA DE INGRESO',
                'A': 'AISLADO' }
train_df.rename(columns = cols_rename, inplace = True)

In addition, each time you move to the next day according to the date of entry, the first column returns to 1:

In [196]:
train_df.loc[train_df['FECHA DE INGRESO'] == '01-01', '3+-99999|a'].iloc[0] == train_df.loc[train_df['FECHA DE INGRESO'] == '01-02', '3+-99999|a'].iloc[0]

True

Therefore, I will name that first column "NUMERO DE TURNO" in reference to the fact that the turns are reset at the beginning of the next day:

In [197]:
col_rename = { '3+-99999|a': 'NUMERO DE TURNO' }
train_df.rename(columns = col_rename, inplace = True)

## 1.2 Removing headers and null rows

We will remove the header that appears every time a new day begins:

In [198]:
train_df = train_df[train_df['NUMERO DE TURNO'].str.contains('FECHA|N°') == False]

In addition, we will remove the last rows of the dataset that do not contain any value in first and last name

In [199]:
train_df = train_df[train_df['NOMBRE Y APELLIDO'].notnull()]

## 1.3 Changing Triage Level data type

For possible machine learning models in the future, the triage level should be int dtype:

In [200]:
train_df['TRIAGE'].dtype

dtype('O')

In [201]:
train_df['TRIAGE'] = train_df['TRIAGE'].str.strip().str.upper()

In [202]:
vals_rename = { 'I': 1, 'II': 2, 'III': 3, 'IV': 4 }
train_df['TRIAGE'] = train_df['TRIAGE'].replace(vals_rename)

Those rows not containing 1, 2, 3, or 4 will be taken as null (using errors coerce)

In [203]:
train_df['TRIAGE'] = pd.to_numeric(train_df['TRIAGE'], downcast = "signed", errors = 'coerce')

In [204]:
train_df['TRIAGE'].dtype

dtype('float64')

## 1.4 Changing Entry Date data type

We will change the object type of the entry date to date type. This makes date manipulation much easier and more readable:

In [205]:
train_df['FECHA DE INGRESO'].dtype

dtype('O')

In [206]:
train_df['FECHA DE INGRESO'] = train_df['FECHA DE INGRESO'] + '-2024'

In [207]:
train_df['FECHA DE INGRESO'] = pd.to_datetime(train_df['FECHA DE INGRESO'], format = '%d-%m-%Y', errors = 'coerce')

In [208]:
train_df['FECHA DE INGRESO'].dtype

dtype('<M8[ns]')

## 1.5 Changing Isolated data type

In [209]:
train_df['AISLADO'].unique()

array([nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'ECOLI METALO'],
      dtype=object)

In [210]:
data = {'AISLADO': [np.nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'A', 'ECOLI METALO']}
mapping = {np.nan: False, 'NO': False, 'No': False,
           'KPC': True, 'ECOLI METALO': True, 'SI': True, 'Si': True, 'A': True}
train_df['AISLADO'] = train_df['AISLADO'].map(mapping)

In [211]:
train_df = train_df[train_df['AISLADO'] != '724']

In [212]:
train_df['AISLADO'] = train_df['AISLADO'].astype(bool)

## 1.6 Generating ALTA column

In [213]:
train_df['DESTINO'].unique()

array([nan, 'alta', 'ALTA', '624', 'ALTA ', 'FUGA', '506', '?', 'INT',
       '715', '721', '512', '717', '820', '622', 'ALT.VOL', '312', '812',
       'HMD/ 315', 'HMD', '316', '301', '806', 'Alta', '802', '809',
       'PISO', '315', '722', '805', 'alta ', 'OBITO', 'Traslado', ' ',
       '    ', '311', '808', 'ALTA VOLUNT', '302', '813', '818', '304',
       'ALTA V ', '814', '613', '619', '602', '615', 'ALTA MEDICA', '314',
       '626', '607', '726', 'ALTA VOL', '926', '922', '920', '917', '916',
       '711', 'DERIVACION', '714', '719', '  ', '724', 'FUGA ', 'DERIV',
       'AL. VOL', '915', '918', 'QUIROFANO', 'ALTA VOLUNT.', '310',
       'TRASLADO ', '928', '919', '616', 'QUIROFANO ', '628', '909',
       '804', 'ALTA VOLUNTARIA', 'DERIVACION ', '720', '803', 'DERIVAC',
       'alTA', '908', '709', '621', '306',
       'ALTA                                                                                                                                                           

In [214]:
train_df = train_df[~train_df['DESTINO'].isin(['?', 'INT', 'HMD/ 315', 'HMD', ' ', '    ', '  ', ])]

In [215]:
train_df['ALTA'] = train_df['DESTINO'].str.contains(
    'alta|obito|traslado|derivacion|AL. VOL|DERIVAC',
    case=False,  
    regex=True   
)
train_df['ALTA'].fillna(True, inplace=True)

Finally, empty and unnamed columns can be visualized. We will remove them:

In [216]:
cols_to_keep = ['NUMERO DE TURNO', 'FECHA DE INGRESO', 'NOMBRE Y APELLIDO', 'MOTIVO DE CONSULTA', 'BOX', 'TRIAGE', 'ENFERMERO', 'MEDICO', 'DESTINO', 'ALTA', 'AISLADO']
train_df = train_df[cols_to_keep]

This is how our dataframe would look at first:

In [217]:
train_df.head()

,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,ALTA,AISLADO
1,1,2024-01-01,LUSI,FIEBRE Y TOS,18,4.0,ERIKA,RODRIGO,NaN,True,False
2,2,2024-01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,4.0,SOLEDAD G,SOLEDAD,alta,True,False
3,3,2024-01-01,BANDERA,TOS,12,4.0,ERIKA,RODRIGO,NaN,True,False
4,4,2024-01-01,TOMINO EDUARDO,HTA,12,4.0,SOLEDAD G,SOLEDAD,ALTA,True,False
5,5,2024-01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,4.0,ERIKA,SOLEDAD,NaN,True,False


In [218]:
def grafico_barras(df, p_x, p_y, title, x_title, y_title, media = None):
    fig = px.bar(df, x=p_x, y=p_y, barmode="group")
    fig.update_layout(title=title, xaxis_title=x_title, yaxis_title=y_title, title_x=0.5)
    fig.update_traces(texttemplate='%{y}', textposition='outside')
    if(media):
        media = df[p_y].mean()
        fig.add_trace(go.Scatter(x=df[p_x],
                                 y=[media] * len(df),
                                 mode='lines',
                                 name='Media',
                                 line=dict(color='red', width=2, dash='dash'),
                                 hovertemplate='%{y:.2f}'))
        fig.add_annotation(
        xref='paper', yref='y',
        x=-0.03, y=media,
        text=f'{media:.2f}',
        showarrow=False,
        font=dict(color='red'))
    fig.show()


In [219]:
def filtros(df, desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    if(desde != None and hasta != None):
        desde = pd.to_datetime(desde, format='%d-%m-%Y', errors='coerce') 
        hasta = pd.to_datetime(hasta, format='%d-%m-%Y', errors='coerce')   
        df = df[(df['FECHA DE INGRESO'] >= desde) & (df['FECHA DE INGRESO'] <= hasta)]
        
    df['FECHA DE INGRESO'] = df['FECHA DE INGRESO'].dt.strftime('%d-%m-%Y')      
       
    if(nombre_y_apellido != None):
        df = df[df['NOMBRE Y APELLIDO'] == nombre_y_apellido]

    if(motivo_de_consulta != None):
        df = df[df['MOTIVO DE CONSULTA'] == motivo_de_consulta]

    if(box != None):
        df = df[df['BOX'] == box]

    if(triage != None):
        df = df[df['TRIAGE'] == triage]

    if(medico != None):
        df = df[df['MEDICO'] == medico]

    if(enfermero != None):
        df = df[df['ENFERMERO'] == enfermero]

    if(alta != None):
        df = df[df['ALTA'] == alta]

    if(aislado != None):
        df = df[df['AISLADO'] == aislado]

    return df

In [220]:
def cant_pacientes_fecha(desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    train_df_acotado = train_df.groupby('FECHA DE INGRESO', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    train_df_acotado = filtros(train_df_acotado, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    return train_df_acotado


In [221]:
df = cant_pacientes_fecha(desde = None, hasta = None,
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='FECHA DE INGRESO', p_y='CANTIDAD DE PACIENTES', 
               x_title='Cantidad de Pacientes por Fecha de Ingreso', y_title='Fecha de Ingreso', title='Cantidad de Pacientes', 
               media=True)


In [222]:
def top_consultas_fecha(desde=None, hasta=None, top = 10, order=None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    top_motivos = train_df.copy(deep=True)
    top_motivos = filtros(top_motivos, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    
    motivos_count = top_motivos['MOTIVO DE CONSULTA'].value_counts()

    top_motivos = motivos_count.nlargest(top).reset_index()
    top_motivos.columns = ['MOTIVO DE CONSULTA', 'CANTIDAD DE CONSULTAS']

    if order == 'asc':
        top_motivos.sort_values(by='CANTIDAD DE CONSULTAS', ascending = True, inplace = True)
        
    return top_motivos

In [223]:
df = top_consultas_fecha(desde = None, hasta=None,
             top = 15, order = 'asc',
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='MOTIVO DE CONSULTA', p_y='CANTIDAD DE CONSULTAS', 
               title='Cantidad de Consultas', x_title='Motivos de Consulta mas Frecuentes', y_title='Motivo de Consulta', 
               media=False)

In [224]:
def cant_pacientes_triage(desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    train_df_acotado = train_df.groupby('TRIAGE', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    train_df_acotado = filtros(train_df_acotado, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    return train_df_acotado


In [225]:
# df = cant_pacientes_triage(desde = None, hasta = None,
#              nombre_y_apellido = None, motivo_de_consulta = None,
#              box = None, triage = None,
#              medico = None, enfermero = None,
#              alta = None, aislado = None)

# grafico_barras(df = df, 
#                p_x='TRIAGE', p_y='CANTIDAD DE PACIENTES', 
#                x_title='Nivel de Triage', y_title='Cantidad de pacientes', title='Cantidad de Pacientes por Nivel de Triage', 
#                media=True)

In [226]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [227]:
def preprocess_text(text):
    # convert to lowercase
    text = str(text).lower()

    # remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])

    # separate by syllables
    tokens = nltk.word_tokenize(text, 'spanish')

    # eliminate common words without meaning
    stop_words = set(stopwords.words('spanish'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization (converting words to their base form)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return ' '.join(lemmatized_tokens)

In [228]:
train_df['MOTIVO DE CONSULTA'] = train_df['MOTIVO DE CONSULTA'].apply(preprocess_text)

display(train_df['MOTIVO DE CONSULTA'])

1                              fiebre tos
2                              fiebre tos
3                                     tos
4                                     hta
5                                  fiebre
6                   sangrado bolsillo mcp
7                                 astenia
8                    hematuria hace 3 dia
9                                  fiebre
10                      dolor rodilla izq
11                           dolor lumbar
12           dolor hueco popliteo derecho
13                                diarrea
14                             odinofagia
15                                sincope
16                                sincope
17                              forunculo
18                       dolor costal izq
20                          palpitaciones
21                              hematuria
22                            dolor pecho
23                                    hta
24                            dolor pecho
25                          palpit

In [229]:
# sintomas_rellenados_por_enfermeros = [sentence.split() for sentence in train_df['MOTIVO DE CONSULTA']]


# # Cargar el modelo preentrenado
# model = KeyedVectors.load_word2vec_format('..\dataset\Pre-Trained-Word2Vec-Files\GoogleNews-vectors-negative300.bin', binary=True)

# # Ahora puedes usar el modelo para encontrar los síntomas más similares
# sintomas_no_entrenados=[]
# for lista_sintomas in sintomas_rellenados_por_enfermeros:
#     print(f"Lista de síntomas: {lista_sintomas}")
#     # Para cada síntoma en la lista...
#     for sintoma in lista_sintomas:
#         print(f"  Síntoma: {sintoma}")
#         # Verificar si el síntoma está en el vocabulario del modelo
#         if sintoma in model.key_to_index:
#             # Encontrar el síntoma más similar
#             similares = model.most_similar(sintoma, topn=1)
#             for sim in similares:
#                 print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
#         else:
#             sintomas_no_entrenados.append(sintoma)
#             #print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")

In [243]:
sintomas_rellenados_por_enfermeros = [sentence.split() for sentence in train_df['MOTIVO DE CONSULTA']]
#print(sintomas_rellenados_por_enfermeros)
#Entreno el modelo usando los sintomas ideales (esto puede variari no solo por cada centro sino por temporada)
#Agegue a la lista que nos pasaron el sintoma tos, habría que evaluar otros posibles casos de eso
sintomas_ideales=[['convulsiones'],
    ['trauma de cráneo'],
    ['dolor torácico'],
    ['dorsal'],
    ['dolor abdominal'],
    ['lumbar'],
    ['cefalea'],
    ['déficit motor'],
    ['disartria'],
    ['afasia'],
    ['pérdida aguda de visión'],
    ['disnea'],
    ['otro dolor en curso'],
    ['sobredosis de fármacos'],
    ['ingesta de tóxicos'],
    ['sangrado digestivo'],
    ['fiebre'],
    ['tos']]
corpus = sintomas_ideales + sintomas_rellenados_por_enfermeros
model = KeyedVectors.load('..\dataset\Pre-Trained-Word2Vec-Files\complete.kv', mmap='r')


sintoma_mas_similar = ''
mayor_similitud = -1

# SintomaIngresado = sintomas_rellenados_por_enfermeros[0][0]
SintomaIngresado = "fiebre "
#if SintomaIngresado in model.wv.key_to_index:
for sintoma_ideal in sintomas_ideales:
    sintoma_ideal = sintoma_ideal[0]
    if sintoma_ideal in model.wv.key_to_index:
        similitud = model.wv.similarity(SintomaIngresado, sintoma_ideal)
    if similitud > mayor_similitud:
        mayor_similitud = similitud
        sintoma_mas_similar = sintoma_ideal
print(f"El síntoma ideal más similar a '{SintomaIngresado}' es '{sintoma_mas_similar}' con una similitud de {mayor_similitud}.")
#else:
 #   print("El síntoma ingresado no está en el vocabulario del modelo.")


# sintomas_no_entrenados=[]
# for lista_sintomas in sintomas_rellenados_por_enfermeros:
#     print(f"Lista de síntomas: {lista_sintomas}")
#     Para cada síntoma en la lista...
#     for sintoma in lista_sintomas:
#         print(f"  Síntoma: {sintoma}")
#         Verificar si el síntoma está en el vocabulario del modelo
#         if sintoma in model.wv.key_to_index:
#             Encontrar el síntoma más similar
#             similares = model.wv.most_similar(sintoma, topn=1)
#             for sim in similares:
#                 print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
#         else:
#             sintomas_no_entrenados.append(sintoma)
#             print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")

TypeError: KeyedVectors.load_word2vec_format() got an unexpected keyword argument 'mmap'

In [231]:

# #Le asigno un valor a cada palabra
# for sintoma in sintomas_rellenados_por_enfermeros:
#     print(f"Síntoma: {sintoma}")
#     similares = model.wv.most_similar(sintoma)
#     for sim in similares:
#         print(f"  - {sim[0]}: {sim[1]}")


# sintomas_rellenados_por_enfermeros = train_df['MOTIVO DE CONSULTA'].tolist()
# #print(sintomas_rellenados_por_enfermeros)
# #Entreno el modelo usando los sintomas ideales (esto puede variari no solo por cada centro sino por temporada)
# #Agegue a la lista que nos pasaron el sintoma tos, habría que evaluar otros posibles casos de eso
# sintomas_ideales=[['convulsiones'],
#     ['trauma de cráneo'],
#     ['dolor torácico'], 
#     ['dorsal'],
#     ['dolor abdominal'], 
#     ['lumbar'],
#     ['cefalea'],
#     ['déficit motor'], 
#     ['disartria'], 
#     ['afasia'],
#     ['pérdida aguda de visión'],
#     ['disnea'],
#     ['otro dolor en curso'],
#     ['sobredosis de fármacos'],
#     ['ingesta de tóxicos'],
#     ['sangrado digestivo'],
#     ['fiebre'],
#     ['tos']]
# model = Word2Vec(sintomas_ideales, vector_size=50, window=5, min_count=1, sg=0)
# # #Le asigno un valor a cada palabra
# # for sintoma in sintomas_rellenados_por_enfermeros:
# #     print(f"Síntoma: {sintoma}")
# #     similares = model.wv.most_similar(sintoma)
# #     for sim in similares:
# #         print(f"  - {sim[0]}: {sim[1]}")

# sintomas_no_relacionados=[]
# for sintoma in sintomas_rellenados_por_enfermeros:
#     # Para cada síntoma en la lista...
#     print(f"  Síntoma: {sintoma}")
#     # Verificar si el síntoma está en el vocabulario del modelo
#     if sintoma in model.wv.key_to_index:
#     # Encontrar el síntoma más similar
#         similares = model.wv.most_similar(sintoma, topn=1)
#         for sim in similares:
#             print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
#     else:
#         sintomas_no_relacionados.append(sintoma)
#             #print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")

In [232]:
print("Cantidad de pacientes: " + str(train_df.shape[0]))
df = top_consultas_fecha(desde = "1-1-2024", hasta="3-3-2024",
             top = 50, order = 'asc',
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='MOTIVO DE CONSULTA', p_y='CANTIDAD DE CONSULTAS', 
               title='Cantidad de Consultas', x_title='Motivos de Consulta mas Frecuentes', y_title='Motivo de Consulta', 
               media=False)

Cantidad de pacientes: 1425


In [233]:
word_vectors = [model.wv[word] for word in model.wv.index_to_key]
print(word_vectors)
X = np.array(word_vectors)

kmeans = KMeans(n_clusters=13)
kmeans.fit(X)

cluster_labels = kmeans.labels_
word_clusters = {word: label for word, label in zip(model.wv.index_to_key, cluster_labels)}

for word, label in word_clusters.items():
    print(f"{word}: Clúster {label}")


[array([ 7.11202483e-06,  9.33861247e-06,  1.18259825e-02,  1.78940371e-02,
       -1.94028877e-02, -1.60645638e-02,  1.42518226e-02,  1.96973849e-02,
       -1.33454595e-02, -9.60635114e-03,  1.58169344e-02, -5.68908406e-03,
       -7.83389993e-03,  1.35560166e-02, -1.13432445e-02, -2.99686729e-03,
        6.73306547e-03,  2.83405650e-03, -1.95729118e-02, -2.11099610e-02,
        1.70080680e-02,  1.27611253e-02,  1.51219210e-02, -5.36428357e-04,
        1.39836641e-02, -6.89244317e-03, -3.83593817e-03,  1.04032429e-02,
       -1.77806187e-02, -7.67366309e-03, -1.62775125e-02, -7.10208900e-04,
        1.75320711e-02, -1.62956044e-02, -5.59876440e-03, -4.01723525e-03,
        1.81091335e-02, -1.10493768e-02,  6.12873817e-04, -1.21914567e-02,
       -1.93757173e-02,  8.33643321e-03, -1.60441492e-02, -8.42115749e-03,
        7.90549035e-04, -1.39184098e-03, -1.68559328e-02,  1.86278615e-02,
        1.06851459e-02,  1.93187334e-02], dtype=float32), array([-1.6079316e-02,  8.8495519e-03, -7

In [234]:
from sklearn.cluster import KMeans
import numpy as np

# Lista de síntomas
sintomas = [
    'Convulsiones',
    'Trauma de Cráneo',
    'Dolor torácico / dorsal',
    'Dolor abdominal / lumbar',
    'Cefalea',
    'Déficit motor',
    'Disartria - afasia',
    'Pérdida aguda de visión',
    'Disnea',
    'Otro dolor en curso',
    'Sobredosis de fármacos / Ingesta de tóxicos',
    'Sangrado Digestivo',
    'Fiebre >38°'
]

# Asumiendo que 'model' es un modelo Word2Vec entrenado
vectores_palabras = [model.wv[sintoma] for sintoma in sintomas if sintoma in model.wv]

# Verificar si la lista de vectores de palabras no está vacía
if vectores_palabras:
    X = np.array(vectores_palabras)

    # El número de clusters debe ser menor o igual al número de síntomas
    kmeans = KMeans(n_clusters=min(13, len(sintomas)))
    kmeans.fit(X)

    etiquetas_clusters = kmeans.labels_
    clusters_palabras = {sintoma: etiqueta for sintoma, etiqueta in zip(sintomas, etiquetas_clusters)}

    for sintoma, etiqueta in clusters_palabras.items():
        print(f"{sintoma}: Clúster {etiqueta}")
else:
    print("Ninguno de los síntomas está en el vocabulario del modelo Word2Vec.")


Ninguno de los síntomas está en el vocabulario del modelo Word2Vec.


In [235]:
#Usar pca para poder graficar y visualizar en dos dimensiones

# Use PCA to reduce the dimensionality of the word vectors to 2D
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)
# Create a DataFrame with the 2D word vectors and their corresponding words and cluster labels
df = pd.DataFrame(word_vectors_2d, columns=['Component 1', 'Component 2'])
df['Word'] = model.wv.index_to_key
df['Cluster Label'] = cluster_labels

# Create a scatter plot of the 2D word vectors, colored by their cluster labels, and with hover text for the words
fig = px.scatter(df, x='Component 1', y='Component 2', color='Cluster Label', hover_data=['Word'])
fig.show()

In [236]:
# Use PCA to reduce the dimensionality of the word vectors to 3D
pca = PCA(n_components=3)
word_vectors_3d = pca.fit_transform(word_vectors)

# Create a DataFrame with the 3D word vectors and their corresponding words and cluster labels
df = pd.DataFrame(word_vectors_3d, columns=['Component 1', 'Component 2', 'Component 3'])
df['Word'] = model.wv.index_to_key
df['Cluster Label'] = cluster_labels

# Create a 3D scatter plot of the word vectors, colored by their cluster labels, and with hover text for the words
fig = go.Figure(data=[go.Scatter3d(
    x=df['Component 1'],
    y=df['Component 2'],
    z=df['Component 3'],
    mode='markers',
    marker=dict(
        size=12,
        color=df['Cluster Label'],                # set color to an array/list of desired values
        colorscale='Viridis',   # choose a colorscale
        opacity=0.8
    ),
    text=df['Word']
)])

# tight layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
fig.show()